# 01.08_processing_2018Zebrafish

整理斑马鱼数据与基因名。

- 当前文件：`analysis/01_preprocessing/01.08_processing_2018Zebrafish.ipynb`
- 原始来源：`Codes/01.08_processing_2018Zebrafish.ipynb`（旧编号仅用于溯源）。
- 运行内核：**python**。
- 导入依赖：`numpy`, `pandas`, `scanpy`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


# Danio rerio

### 1.读取数据

In [ ]:
import scanpy as sc
import pandas as pd

# 正确读取并转置表达矩阵
adata_dare = sc.read_h5ad("/share/home/zhangze/zz/NeuralOrigin/Data/01.RawData/PublicData/2018Zebrafish/WagnerScience2018.h5ad")
adata_dare

In [ ]:
adata_dare[adata_dare.obs["TimeID"] == "24hpf"].obs

In [ ]:
import scanpy as sc

# 假设你的 adata 已经加载
adata = adata_dare[adata_dare.obs["TimeID"] == "24hpf"].copy()
adata

In [ ]:
print(len(adata.obs["ClusterName"].value_counts()))
print(adata.obs["ClusterName"].value_counts())

In [ ]:
print(len(adata.obs["TissueName"].value_counts()))
print(adata.obs["TissueName"].value_counts())

In [ ]:
adata = adata[adata.obs["TissueName"] != 'NaN'].copy()
adata

In [ ]:
print(len(adata.obs["TissueName"].value_counts()))
print(adata.obs["TissueName"].value_counts())

In [ ]:
type_map = {
    "Hindbrain / Spinal Cord":"Neural Posterior",
    "Mesoderm":"Mesoderm",
    "Forebrain / Optic":"Neural Anterior",
    "Midbrain":"Neural Mid",
    "Epidermal":"Epidermal",
    "Neural Crest":"Neural Crest",
    "Endoderm":"Endoderm",
    "Germline":"Germline",
}

In [ ]:
adata.obs['broad_cell_type'] = adata.obs['TissueName'].map(type_map)
adata

In [ ]:
print(len(adata.obs["broad_cell_type"].value_counts()))
print(adata.obs["broad_cell_type"].value_counts())

### 2.基因名映射

In [ ]:
adata.var

In [ ]:
em = pd.read_csv("/share/home/zhangze/zz/NeuralOrigin/Data/01.RawData/PublicData/2018Zebrafish/Dare.emapper.annotations.tsv", sep='\t')
em

In [ ]:
# 确保 evalue 为浮点型
em['evalue'] = pd.to_numeric(em['evalue'], errors='coerce')

# 只保留每个 Preferred_name 对应 evalue 最小的那一行
idx = em.groupby('Preferred_name')['evalue'].idxmin()

# 得到筛选后的 DataFrame
mapping_df = em.loc[idx, ['Preferred_name', 'query', 'evalue']]
mapping_df

In [ ]:
# 可将其转为字典（Preferred_name -> query）
pref2query = dict(zip(mapping_df['Preferred_name'], mapping_df['query']))

In [ ]:
# 1. 转为 DataFrame
var_df = adata.var.reset_index()  # 会把 var_names 作为一列 index
var_df

In [ ]:
# 2. 构建用于忽略大小写匹配的 DataFrame
var_df['gene_name_upper'] = var_df['index'].str.upper()
var_df

In [ ]:
var_df = var_df.drop_duplicates(subset=['gene_name_upper'])
var_df

In [ ]:
# 3. 构建 pref2query DataFrame，也小写处理
pref_df = pd.DataFrame(list(pref2query.items()), columns=['Preferred_name', 'query'])
pref_df['Preferred_name_upper'] = pref_df['Preferred_name'].str.upper()
pref_df

In [ ]:
pref_df = pref_df.drop_duplicates(subset=['Preferred_name_upper'])
pref_df

In [ ]:
# 4. 备份原有基因名
var_df['gene_name_orig'] = var_df['index']
var_df

In [ ]:
# 5. 合并，左连接保证全部保留
merged = pd.merge(var_df, pref_df, left_on='gene_name_upper', right_on='Preferred_name_upper', how='left')
merged

In [ ]:
# 6. 用 query 替换原基因名（有匹配则替换，否则保留原值）
merged['index'] = merged['query'].combine_first(merged['index'])
merged

In [ ]:
# 7. 用新的 index 更新 adata.var_names
adata.var_names = merged['index'].values
adata

In [ ]:
# 8. 把备份基因名也加回去
adata.var['gene_name_orig'] = merged['gene_name_orig'].values
adata

In [ ]:
adata.var

In [ ]:
# 查找重复的基因名
duplicates = adata.var_names[adata.var_names.duplicated()]
# 输出重复的基因名
print("重复的基因名：", duplicates)
# 删除重复的基因（保留第一个出现的）
adata = adata[:, ~adata.var_names.duplicated()]
adata.var

In [ ]:
# 基因重命名，更换_为-
adata.var_names = adata.var_names.str.replace('_', '-', regex=False)
adata.var

### 3.降维聚类

In [ ]:
# 细胞名
adata.obs

In [ ]:
# 矩阵内容
# Values were log1p-normalized, mean-centered and scaled, and filtered for highly variable genes using the same procedure described above for downstream dimensionality reduction and visualization (e.g., PCA) using Scanpy. 
adata.X.toarray()

In [ ]:
max(adata.X.toarray()[0])

In [ ]:
adata

In [ ]:
# 标准化和 log 转换
sc.pp.normalize_total(adata, target_sum=1e4)    # 对应 scale.factor=10000
sc.pp.log1p(adata)
# 高度变异基因选择（如果需要）
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
print(adata)

In [ ]:
# 保存原始数据
adata.raw = adata.copy()

In [ ]:
# 取高可变基因
adata = adata[:, adata.var.highly_variable]

# PCA
# sc.tl.pca(adata, svd_solver='arpack')
sc.tl.pca(
    adata,
    n_comps=30,          # 计算 30 个主成分（PCs）
    svd_solver='arpack',  # 默认方法
)

# 计算邻居图
# sc.pp.neighbors(adata)
sc.pp.neighbors(
    adata,
    n_neighbors=50,      # 明确指定 k=50
    n_pcs=30,            # 使用前 30 个 PCs
    method='umap',       # 默认基于 UMAP 的 KNN
    random_state=42      # 可复现性
)

# 计算 UMAP
# sc.tl.umap(adata)
sc.tl.louvain(
    adata,
    resolution=1.0,      # 明确指定分辨率=1
    random_state=42,     # 可复现性
    key_added='louvain'  # 聚类结果存储到 adata.obs['louvain']
)

# UMAP 可视化（基于 Louvain 聚类）
sc.tl.umap(
    adata,
    random_state=42,
    n_components=2           # 2D 可视化
)

# 绘制 UMAP 图（按 Louvain 聚类着色）
sc.pl.umap(
    adata,
    color="louvain",         # 使用 Louvain 聚类结果
    legend_loc="on data",    # 将簇标签显示在图上
    frameon=False,
    title="UMAP (Louvain clustering)"
)

In [ ]:
adata

In [ ]:
# 可视化 UMAP
sc.pl.umap(adata, color=['louvain'])

In [ ]:
# 可视化 UMAP
sc.pl.umap(adata, color=['ClusterName'])

In [ ]:
# 可视化 UMAP
sc.pl.umap(adata, color=['broad_cell_type'])

In [ ]:
# TSNE 可视化
sc.tl.tsne(adata, random_state=42)
sc.pl.tsne(adata, color='broad_cell_type')

In [ ]:
adata = adata.raw.to_adata()
adata

In [ ]:
adata

In [ ]:
import numpy as np
print("Percentiles:", np.percentile(adata.X.data, [0, 25, 50, 75, 100]))

In [ ]:
# 保存adata数据
raw_dare_path = "/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/Dare.normalized.h5ad"
adata.write(raw_dare_path)

In [ ]:
# 导出基因id
adata.var_names.to_series().to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/Dare.genes.txt', index=False, header=False)